In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from db_config import DB_URL

engine = create_engine(DB_URL)

In [2]:
df = pd.read_sql("""
    SELECT f.*, d.internet, d.famsup, d.health
    FROM fact_grades f
    JOIN dim_student d USING (student_id)
""", engine)

In [ ]:
# COMPOSITE RISK SCORE — weighted factors
# Each component is normalised to 0–1 first, then weighted
# Higher score = higher dropout risk

In [ ]:
# Component 1: Low final grade (weight 35%)
# G3 is 0-20; invert so low grade = high risk
grade_risk = np.clip(1 - (df["g3"].values / 20), 0, 1) # pyright: ignore[reportOperatorIssue]

In [4]:
# Component 2: Past failures (weight 30%)
# failures is 0-4; normalise to 0-1
failure_risk = np.clip(df["failures"].values / 4, 0, 1)  #type: ignore

In [5]:
# Component 3: High absences (weight 20%)
# Cap at 30+ absences = max risk
absence_risk = np.clip(df["absences"].values / 30, 0, 1) #type: ignore

In [6]:
# Component 4: Declining grade trend (weight 15%)
# grade_trend is G3-G1; range typically -10 to +10
# Invert and normalise: negative trend = higher risk
trend_risk = np.clip(-df["grade_trend"].values / 10, 0, 1) #type: ignore

In [7]:
# Weighted composite score (0–100)
risk_score = np.round(
    (grade_risk   * 35 +
     failure_risk * 30 +
     absence_risk * 20 +
     trend_risk   * 15),
    2
)

In [8]:
df["risk_score"] = risk_score

In [12]:
# at_risk flag — top quartile of risk scores
threshold        = np.percentile(risk_score, 75)
df["at_risk"]    = np.where(risk_score >= threshold, True, False)

In [13]:
print("=== Risk score distribution ===")
for p in [25, 50, 75, 90]:
    print(f"  P{p}: {np.percentile(risk_score, p):.1f}")

print(f"\nAt-risk students  : {df['at_risk'].sum()} "
      f"({df['at_risk'].mean()*100:.1f}%)")
print(f"Risk threshold    : {threshold:.1f} (75th percentile)")

=== Risk score distribution ===
  P25: 13.1
  P50: 18.4
  P75: 25.1
  P90: 40.0

At-risk students  : 261 (25.0%)
Risk threshold    : 25.1 (75th percentile)


In [14]:
# Show top 10 highest-risk students
print("\n=== Top 10 highest-risk students ===")
print(
    df[["student_id","g1","g2","g3","failures",
        "absences","grade_trend","risk_score"]]
    .sort_values("risk_score", ascending=False)
    .head(10)
    .to_string(index=False)
)


=== Top 10 highest-risk students ===
 student_id  g1  g2  g3  failures  absences  grade_trend  risk_score
        174   8   7   0         3         0           -8       69.50
       1006   8   0   0         3         0           -8       69.50
        147   6   7   0         3         0           -6       66.50
        151   6   5   0         3         0           -6       66.50
        559  11   9   0         2         0          -11       65.00
        154   5   0   0         3         0           -5       65.00
        145   5   0   0         3         0           -5       65.00
        131  12   0   0         2         0          -12       65.00
         19   6   5   5         3        16           -1       60.92
        217   6   6   4         2        22           -2       60.67


In [15]:
# Write updated risk scores back to PostgreSQL
with engine.begin() as conn:
    for _, row in df.iterrows():
        conn.execute(text("""
            UPDATE fact_grades
            SET risk_score = :rs,
                at_risk    = :ar
            WHERE student_id = :sid
              AND course     = :course
        """), {
            "rs"    : float(row["risk_score"]),
            "ar"    : bool(row["at_risk"]),
            "sid"   : int(row["student_id"]),
            "course": row["course"]
        })

print("\nRisk scores written back to fact_grades ✓")


Risk scores written back to fact_grades ✓
